# FaceForensics Xception 추론 환경 점검

이 노트북은 VS Code 가상환경과 기존 FaceForensics Xception 추론이 동작하는지만 점검합니다. 모델 구조와 기존 추론 코드는 수정하지 않습니다.

## 1. 현재 Jupyter 커널 확인

현재 노트북이 실제로 어떤 Python 인터프리터와 버전으로 실행되는지 확인합니다.

In [5]:
import sys

print(f'Python 실행 경로: {sys.executable}')
print(f'Python 버전: {sys.version}')

Python 실행 경로: c:\Users\user\AppData\Local\Programs\Python\Python313\python.exe
Python 버전: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]


## 2. 핵심 라이브러리 버전 확인

Xception 추론에 필요한 PyTorch, Torchvision, OpenCV, NumPy의 설치 버전을 출력합니다.

In [6]:
import cv2
import numpy as np
import torch
import torchvision

print(f'torch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'opencv: {cv2.__version__}')
print(f'numpy: {np.__version__}')

torch: 2.12.0+cpu
torchvision: 0.27.0+cpu
opencv: 4.11.0
numpy: 2.2.5


## 3. CUDA 및 GPU 확인

CUDA 사용 가능 여부를 확인하고, 사용할 수 있으면 첫 번째 GPU 이름을 출력합니다.

In [7]:
CUDA_AVAILABLE = torch.cuda.is_available()
print(f'CUDA 사용 가능: {CUDA_AVAILABLE}')
if CUDA_AVAILABLE:
    print(f'GPU 이름: {torch.cuda.get_device_name(0)}')
else:
    print('GPU 이름: CUDA를 사용할 수 없음')

CUDA 사용 가능: False
GPU 이름: CUDA를 사용할 수 없음


## 4. FinGuard 프로젝트 루트 탐색 및 경로 등록

현재 작업 폴더와 상위 폴더에서 `FinGuard` 루트를 자동으로 찾습니다. 해당 폴더가 없으면 FaceForensics 소스 루트를 대체 경로로 사용합니다.

In [8]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if candidate.name.lower() == 'finguard':
            return candidate
        for child in candidate.glob('*'):
            if child.is_dir() and child.name.lower() == 'finguard':
                return child
    for candidate in candidates:
        if (candidate / 'video' / 'classification' / 'detect_from_video.py').is_file():
            print('FinGuard 폴더를 찾지 못해 FaceForensics 소스 루트를 사용합니다.')
            return candidate
    raise FileNotFoundError('FinGuard 또는 FaceForensics 프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INFERENCE_PYTHON = Path(sys.executable)

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'sys.path 등록됨: {str(PROJECT_ROOT) in sys.path}')

FinGuard 폴더를 찾지 못해 FaceForensics 소스 루트를 사용합니다.
프로젝트 루트: C:\Users\user\Desktop\minkyung\FaceForensics
sys.path 등록됨: True


## 5. 모델 및 테스트 영상 경로 설정

가중치와 테스트 영상은 별도 변수로 관리합니다. 환경 변수로 경로를 지정하지 않으면 아래 기본 경로를 사용하므로, 실제 파일 위치에 맞게 수정합니다.

In [9]:
import os

MODEL_WEIGHTS_PATH = Path(os.environ.get(
    'FINGUARD_MODEL_WEIGHTS_PATH',
    PROJECT_ROOT / 'video' / 'face_detection' / 'xception' / 'all_c23.p',
))
TEST_VIDEO_PATH = Path(os.environ.get(
    'FINGUARD_TEST_VIDEO_PATH',
    PROJECT_ROOT / 'video' / 'sample1.mp4',
))
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'xception_inference'

print(f'모델 가중치: {MODEL_WEIGHTS_PATH}')
print(f'테스트 영상: {TEST_VIDEO_PATH}')
print(f'출력 폴더: {OUTPUT_PATH}')

모델 가중치: C:\Users\user\Desktop\minkyung\FaceForensics\video\face_detection\xception\all_c23.p
테스트 영상: C:\Users\user\Desktop\minkyung\FaceForensics\video\sample1.mp4
출력 폴더: C:\Users\user\Desktop\minkyung\FaceForensics\outputs\xception_inference


## 6. 입력 파일 존재 여부 확인

추론 전에 모델 가중치와 테스트 영상이 실제로 존재하는지 확인합니다.

In [10]:
for label, path in {'모델 가중치': MODEL_WEIGHTS_PATH, '테스트 영상': TEST_VIDEO_PATH}.items():
    print(f"{label} 존재: {path is not None and path.is_file()} | {path or '아직 지정되지 않음'}")

if MODEL_WEIGHTS_PATH is None or not MODEL_WEIGHTS_PATH.is_file() or not TEST_VIDEO_PATH.is_file():
    raise FileNotFoundError('모델 가중치는 FINGUARD_MODEL_WEIGHTS_PATH 환경 변수로 지정하고, 테스트 영상 경로를 확인하세요.')

모델 가중치 존재: True | C:\Users\user\Desktop\minkyung\FaceForensics\video\face_detection\xception\all_c23.p
테스트 영상 존재: True | C:\Users\user\Desktop\minkyung\FaceForensics\video\sample1.mp4


## 7. 기존 추론 스크립트 인자 확인

실행 전에 기존 `detect_from_video.py`가 제공하는 argparse 옵션을 출력합니다. 이 스크립트의 실제 옵션은 `--video_path`, `--model_path`, `--output_path`, `--cuda`입니다.

In [11]:
import subprocess

print('Inference Python: {}'.format(INFERENCE_PYTHON))

Inference Python: c:\Users\user\AppData\Local\Programs\Python\Python313\python.exe


## 8. 기존 FaceForensics Xception 추론 실행

기존 `detect_from_video.py`를 수정하지 않고 호출합니다. CUDA가 사용 가능할 때만 `--cuda` 옵션을 전달합니다.

In [12]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
RESULT_PATH = OUTPUT_PATH / (TEST_VIDEO_PATH.stem + '.json')
ANNOTATED_VIDEO_PATH = OUTPUT_PATH / (TEST_VIDEO_PATH.stem + '.avi')

command = [
    str(INFERENCE_PYTHON), '-m', 'video.classification.detect_from_video',
    '--video_path', str(TEST_VIDEO_PATH),
    '--model_path', str(MODEL_WEIGHTS_PATH),
    '--output_path', str(OUTPUT_PATH),
    '--result_path', str(RESULT_PATH),
]
if CUDA_AVAILABLE:
    command.append('--cuda')

print('실행 명령:')
print(' '.join(f'\"{part}\"' if ' ' in part else part for part in command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

import json
with RESULT_PATH.open(encoding='utf-8') as result_file:
    result = json.load(result_file)

print(f"Video fake score: {result['scores']['video_fake_score']:.4f}")
print(f"Confidence score: {result['scores']['confidence_score']:.4f}")
print(f"Risk level: {result['risk']['risk_level']}")
print(f"Decision: {result['risk']['decision']}")
print(f"Reasons: {', '.join(result['risk']['reason_codes'])}")
print(f"JSON saved to: {RESULT_PATH}")
print(f"Annotated video saved to: {result['artifacts']['annotated_video_path']}")
print(f"Processing time: {result['processing']['processing_time_ms'] / 1000:.2f} sec")

실행 명령:
c:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m video.classification.detect_from_video --video_path C:\Users\user\Desktop\minkyung\FaceForensics\video\sample1.mp4 --model_path C:\Users\user\Desktop\minkyung\FaceForensics\video\face_detection\xception\all_c23.p --output_path C:\Users\user\Desktop\minkyung\FaceForensics\outputs\xception_inference --result_path C:\Users\user\Desktop\minkyung\FaceForensics\outputs\xception_inference\sample1.json
Video fake score: 0.9325
Confidence score: 0.7810
Risk level: HIGH
Decision: BLOCK
Reasons: VIDEO_DEEPFAKE_HIGH
JSON saved to: C:\Users\user\Desktop\minkyung\FaceForensics\outputs\xception_inference\sample1.json
Annotated video saved to: C:\Users\user\Desktop\minkyung\FaceForensics\outputs\xception_inference\sample1.avi
Processing time: 63.07 sec
